# ⚡ XGBoost GLM Residual Boosting: GPU Trees on Linear Log-Odds (Kaggle PGS S6E11)

Welcome to the **XGBoost GLM Residual Boosting Pipeline** for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🔬 Architecture & Motivation
1. In our previous breakthrough, **GLM Residual LightGBM** achieved our new competition best score of **`0.94782`** on the Public Leaderboard (and **`0.51306`** PR-AUC).
2. Standalone **XGBoost GPU** was our highest-performing individual model family (**`0.94715`** OOF ROC-AUC vs LightGBM's `0.94700`).
3. In this notebook, we apply the exact same mathematical paradigm to **XGBoost**:
   - An L2-regularized **Logistic Regression** calculates the globally calibrated linear log-odds:
     $$z_i = \mathbf{w}^T \mathbf{x}_i + b = \log\left( \frac{p_i}{1 - p_i} \right)$$
   - We inject $z_i$ into XGBoost via `xgb.DMatrix(..., base_margin=z_i)`.
   - XGBoost's GPU histogram tree booster (`tree_method='hist'`, `device='cuda'`, `max_depth=3`) then fits **only the non-linear residuals and feature interaction terms**:
     $$P(y_i = 1 | \mathbf{x}_i) = \sigma\left( z_i + \sum_{m=1}^M \eta f_m(\mathbf{x}_i) \right)$$
4. This combines global convexity from convex optimization with the split power of GPU gradient boosting!

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.special import expit

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"XGBoost Version: {xgb.__version__}")
print(f"Compute Acceleration Device: {device}")

In [ ]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    
    # Model Hyperparameters
    MAX_DEPTH = 3
    LEARNING_RATE = 0.03
    N_ESTIMATORS = 1500
    EARLY_STOPPING_ROUNDS = 50
    SUBSAMPLE = 0.8
    COLSAMPLE_BYTREE = 0.8
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")

In [ ]:
print("Loading datasets...")
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

features = [c for c in train_df.columns if c not in ['id', 'is_fraud']]
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
num_cols = [c for c in features if c not in cat_cols]
bool_cols = [
    'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

# 1. Categorical formatting for XGBoost enable_categorical=True
X_gbdt = train_df[features].copy()
X_test_gbdt = test_df[features].copy()

for col in cat_cols:
    X_gbdt[col] = X_gbdt[col].astype('category')
    X_test_gbdt[col] = X_test_gbdt[col].astype('category')

for b in bool_cols:
    X_gbdt[b] = X_gbdt[b].astype(int)
    X_test_gbdt[b] = X_test_gbdt[b].astype(int)

# 2. Pipeline for Logistic Regression
lr_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

y = train_df['is_fraud'].astype(int).values

print(f"Train samples: {len(train_df):,}, Test samples: {len(test_df):,}")
print(f"Fraud Rate: {y.mean()*100:.3f}% ({y.sum():,} fraud txns)")

In [ ]:
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

oof_lr_probs = np.zeros(len(train_df), dtype=np.float64)
oof_xgb_boost_probs = np.zeros(len(train_df), dtype=np.float64)
test_xgb_boost_probs = np.zeros(len(test_df), dtype=np.float64)

fold_scores = []

print("=" * 70)
print(f"Starting 5-Fold XGBoost GLM Residual Boosting (device={device})...")
print("=" * 70)

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y), 1):
    fold_t0 = time.time()
    
    # ------------------ Phase A: Fit Logistic Regression ------------------
    X_tr_lr = lr_preprocessor.fit_transform(train_df.iloc[train_idx][features])
    X_va_lr = lr_preprocessor.transform(train_df.iloc[val_idx][features])
    X_te_lr = lr_preprocessor.transform(test_df[features])
    y_tr, y_va = y[train_idx], y[val_idx]
    
    lr = LogisticRegression(C=0.1, max_iter=1000, random_state=Config.RANDOM_STATE)
    lr.fit(X_tr_lr, y_tr)
    
    # Linear log-odds margin z = w^T x + b
    z_tr = lr.decision_function(X_tr_lr)
    z_va = lr.decision_function(X_va_lr)
    z_te = lr.decision_function(X_te_lr)
    
    oof_lr_probs[val_idx] = expit(z_va)
    auc_lr = roc_auc_score(y_va, oof_lr_probs[val_idx])
    
    # ------------------ Phase B: Fit XGBoost on Residuals ------------------
    X_tr_g = X_gbdt.iloc[train_idx]
    X_va_g = X_gbdt.iloc[val_idx]
    
    # DMatrix with base_margin set to linear log-odds
    dtrain = xgb.DMatrix(X_tr_g, label=y_tr, base_margin=z_tr, enable_categorical=True)
    dval = xgb.DMatrix(X_va_g, label=y_va, base_margin=z_va, enable_categorical=True)
    dtest = xgb.DMatrix(X_test_gbdt, base_margin=z_te, enable_categorical=True)
    
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': Config.MAX_DEPTH,
        'learning_rate': Config.LEARNING_RATE,
        'subsample': Config.SUBSAMPLE,
        'colsample_bytree': Config.COLSAMPLE_BYTREE,
        'tree_method': 'hist',
        'device': device,
        'random_state': Config.RANDOM_STATE + fold
    }
    
    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=Config.N_ESTIMATORS,
        evals=[(dval, 'val')],
        early_stopping_rounds=Config.EARLY_STOPPING_ROUNDS,
        verbose_eval=False
    )
    
    # Predict final calibrated probabilities (XGBoost applies sigmoid to base_margin + tree_sum)
    val_probs = booster.predict(dval)
    oof_xgb_boost_probs[val_idx] = val_probs
    test_xgb_boost_probs += booster.predict(dtest) / Config.N_SPLITS
    
    auc_boost = roc_auc_score(y_va, val_probs)
    pr_boost = average_precision_score(y_va, val_probs)
    fold_scores.append((auc_boost, pr_boost))
    
    gain = auc_boost - auc_lr
    print(f"Fold {fold} | Best Iter: {booster.best_iteration:4d} | LR AUC: {auc_lr:.5f} -> XGB Boosted AUC: {auc_boost:.5f} (+{gain:.5f}) | PR-AUC: {pr_boost:.5f} ({time.time()-fold_t0:.1f}s)")

total_cv_time = time.time() - start_time
print("=" * 70)
print(f"All 5 Folds Completed in {total_cv_time:.1f}s!")

In [ ]:
overall_lr_auc = roc_auc_score(y, oof_lr_probs)
overall_boost_auc = roc_auc_score(y, oof_xgb_boost_probs)
overall_boost_pr = average_precision_score(y, oof_xgb_boost_probs)

mean_fold_auc = np.mean([s[0] for s in fold_scores])
std_fold_auc = np.std([s[0] for s in fold_scores])

print("-" * 65)
print("XGBoost GLM Residual Boosting Summary:")
print(f"Mean Fold ROC-AUC:                {mean_fold_auc:.5f} ± {std_fold_auc:.5f}")
print(f"Overall OOF ROC-AUC:              {overall_boost_auc:.5f} (+{overall_boost_auc - overall_lr_auc:.5f} vs LR)")
print(f"Overall OOF PR-AUC:               {overall_boost_pr:.5f}")
print("-" * 65)
print(f"Comparison vs Standalone Benchmarks:")
print(f"  Standalone LightGBM:            0.94700")
print(f"  Standalone XGBoost:             0.94715")
print(f"  GLM Residual LightGBM:          0.94708 (LB: 0.94782)")
print(f"  XGBoost GLM Residual Boosting:  {overall_boost_auc:.5f}")
print("-" * 65)

# Save OOF predictions
np.save(Config.OUTPUT_DIR / 'oof_xgb_glm_boost.npy', oof_xgb_boost_probs)
print("Saved oof_xgb_glm_boost.npy")

# Plot Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr_lr, tpr_lr, _ = roc_curve(y, oof_lr_probs)
fpr_boost, tpr_boost, _ = roc_curve(y, oof_xgb_boost_probs)

axes[0].plot(fpr_lr, tpr_lr, color='gray', linestyle='--', label=f'Baseline LR (AUC = {overall_lr_auc:.5f})')
axes[0].plot(fpr_boost, tpr_boost, color='#e65100', lw=2, label=f'XGB GLM Boosted (AUC = {overall_boost_auc:.5f})')
axes[0].plot([0, 1], [0, 1], color='lightgray', linestyle=':')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (XGBoost GLM Residual Boosting)')
axes[0].legend(loc='lower right')

prec_boost, rec_boost, _ = precision_recall_curve(y, oof_xgb_boost_probs)
axes[1].plot(rec_boost, prec_boost, color='#0277bd', lw=2, label=f'XGB Boosted PR (AUC = {overall_boost_pr:.5f})')
axes[1].axhline(y=y.mean(), color='gray', linestyle='--', label=f'Base Rate ({y.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (5-Fold OOF)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('xgb_glm_boost_diagnostics.png', dpi=150)
plt.show()

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': test_xgb_boost_probs
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'xgb_glm_boost_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission successfully generated and verified at: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))